# Phase 3: Defense Evaluation (Base Model & Privacy Filter)
This notebook is currently set up to ONLY run the local evaluation script. 
It will test the Base Qwen 3B model and the OpenAI Privacy Filter baseline. It will safely skip the DPO model if it hasn't been trained yet.

## 0. Setup Repository and Datasets
This cell is safe to run multiple times. It will clone the repo and link your Kaggle dataset.

In [1]:
import os
import shutil

repo_dir = "/kaggle/working/VDT-PII-Mitigate"
if os.path.exists(repo_dir):
    shutil.rmtree(repo_dir)

!git clone https://github.com/no1ceboy/VDT-PII-Mitigate
%cd VDT-PII-Defense

# Link the dataset safely
dataset_target = "/kaggle/working/VDT-PII-Mitigate/datasets"
if os.path.islink(dataset_target) or os.path.exists(dataset_target):
    os.system(f"rm -rf {dataset_target}")

# Update the source path below to match your Kaggle dataset path!
!ln -s /kaggle/input/datasets/no1ceboy/medical-pii {dataset_target}

!pip install -r requirements.txt
!pip install accelerate bitsandbytes peft trl datasets torch torchvision torchaudio transformers wandb
!pip install git+https://github.com/openai/privacy-filter.git

Cloning into 'VDT-PII-Mitigate'...
remote: Enumerating objects: 116, done.
remote: Counting objects: 100% (116/116), done.
remote: Compressing objects: 100% (91/91), done.
remote: Total 116 (delta 23), reused 108 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (116/116), 2.88 MiB | 13.15 MiB/s, done.
Resolving deltas: 100% (23/23), done.
/kaggle/working/VDT-PII-Defense
  Cloning https://github.com/openai/privacy-filter.git to /tmp/pip-req-build-vosx_tvq
  Running command git clone --filter=blob:none --quiet https://github.com/openai/privacy-filter.git /tmp/pip-req-build-vosx_tvq
  Resolved https://github.com/openai/privacy-filter.git to commit f7f00ca7fb869683eb732c010299d901457f19c3
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 1. Setup API Keys & WandB (Optional for Evaluation)
Load API keys and weights & biases.

In [2]:
import os
import wandb
from kaggle_secrets import UserSecretsClient

try:
    user_secrets = UserSecretsClient()
    os.environ['OPENROUTER_API_KEY'] = user_secrets.get_secret("OPENROUTER_API_KEY")
    os.environ['GOOGLE_API_KEY'] = user_secrets.get_secret("GOOGLE_API_KEY")
except:
    print("API keys not found, but they are not required for local evaluation.")

try:
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wandb_key)
    print("Successfully logged into WandB!")
except Exception:
    print("WANDB_API_KEY not found in secrets.")


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: dangthinhtuongminh (dangthinhtuongminh-hanoi-university-of-science-and-techn) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Successfully logged into WandB!


## 2 & 3. Attack & Training (COMMENTED OUT)
These are currently commented out so you can just run the evaluation phase right now.

In [3]:
!python -m src.evaluate_defense \
    --base_model "Qwen/Qwen2.5-1.5B-Instruct" \
    --dpo_model_path "/kaggle/input/models/no1ceboy/vdt-pii-dpo/pytorch/default/3/DPO" \
    --ogpsa_model_path "/kaggle/input/models/no1ceboy/vdt-pii-dpo/pytorch/default/3/OGPSA" \
    --test_source "hf" \
    --hf_offset 2000 \
    --limit 100 \
    --output_file "results/defense_results_detailed.json" \
    --methods Prompt_Defense

usage: evaluate_defense.py [-h] [--model_path MODEL_PATH]
                           [--dpo_model_path DPO_MODEL_PATH]
                           [--ogpsa_model_path OGPSA_MODEL_PATH]
                           [--base_model BASE_MODEL]
                           [--test_source {hf,local}] [--hf_offset HF_OFFSET]
                           [--limit LIMIT] [--output_file OUTPUT_FILE]
evaluate_defense.py: error: unrecognized arguments: --methods Prompt_Defense


## 4. Evaluate Defenses
Evaluate the Attack Success Rate (ASR) of the Base model vs. Baseline Privacy Filter locally.

In [4]:
# !python -m src.evaluate_defense